# Mugello · ¿Cuándo dispara cada detector?

**9 de diciembre de 2019 · Estación FNVD · Canal vertical · Registro archivado de INGV**

Vamos a seguir una grabación desde la lectura hasta el disparo. Los tres
métodos buscan el mismo cambio: un aumento de potencia respecto al fondo.
Al final compararemos sus disparos con la llegada P que marcó un analista.

Ejecuta las celdas en orden con **Python (seismic-wavelets-review)**. La última abre
el resumen para proyectar. Todo se calcula a partir de archivos locales.

## 1 · Qué señal estamos leyendo

El archivo MiniSEED guarda 90 segundos de la componente vertical,
muestreados a 100 Hz. La amplitud está en **cuentas digitales** del instrumento;
no se convirtió a velocidad ni a aceleración.

`load_demo_case` comprueba la huella SHA-256 del archivo original y del recorte,
y verifica que las muestras del recorte estén intactas. **P** y **S** son
marcas manuales de la llegada de dos ondas a la estación. Se usan después
para interpretar el resultado, no para calcular los disparos.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "scripts/demo_mugello_beamer.py").is_file():
    raise RuntimeError("Abre la raíz del repositorio en VS Code.")
if Path(sys.prefix).resolve() != (root / ".venv").resolve():
    raise RuntimeError("Selecciona como kernel la .venv de este clon.")

sys.path.insert(0, str(root / "scripts"))
print("Intérprete:", sys.executable)

In [ ]:
from demo_mugello_beamer import (
    LIVE_DEST,
    export_demo_report,
    load_demo_case,
    open_presentation,
    plot_crossings,
    plot_demo,
    plot_preparation,
    run_demo_detectors,
)

from seismic_wavelets.pilot import FS, HOP, OFFSET, PROTOCOL, TIMING, WIDTH, prepare

trace, waveform_hash, picks, thresholds = load_demo_case()
sampling_hz = trace.stats.sampling_rate
print(f"{trace.id} · {sampling_hz:g} Hz · {trace.stats.npts:,} muestras")
print("Inicio UTC:", trace.stats.starttime)
print("P manual:", trace.stats.starttime + picks["P"])
print("S manual:", trace.stats.starttime + picks["S"])
print("SHA-256 del recorte:", waveform_hash)

## 2 · Qué cambia al preparar la señal

**Eje horizontal:** segundos desde el comienzo del registro. **Eje vertical:**
amplitud en cuentas digitales, con la misma escala en ambos paneles.

- **A · Registro original.** La curva azul muestra las muestras originales.
  La línea naranja de trazos es la media del primer segundo, sombreado en amarillo.
- **B · Señal preparada.** La curva morada muestra las muestras que quedan
  después de restar esa media. El tramo sombreado está vacío porque se descarta.

El gráfico solo enseña los primeros cuatro segundos. El cálculo usa los **89
segundos restantes** y conserva el tiempo original de cada muestra.

In [ ]:
from io import BytesIO

from IPython.display import Image, display
from matplotlib import pyplot as plt


def display_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=155, facecolor="white")
        display(Image(data=buffer.getvalue(), width=960))
    plt.close(figure)


samples = prepare(trace.data)
print(f"Se descartan {OFFSET} muestras; quedan {len(samples) / FS:g} s para el análisis.")
display_figure(plot_preparation(trace, samples))

## 3 · Qué mide cada detector

Piensa en una ventana móvil de **0,64 s** que avanza **0,05 s** cada vez.
En cada paso el programa mide la potencia de la ventana actual y la divide
por la potencia media de un conjunto de ventanas de fondo anteriores.
Ninguna de esas ventanas de fondo comparte muestras con la ventana actual.

| Curva | Qué cuenta como potencia |
| :--- | :--- |
| **Azul continua · STA/LTA** | Promedio de las muestras al cuadrado. Usa toda la señal de la ventana. |
| **Naranja de trazos · STFT** | Potencia de Fourier en la banda de 3,125 a menos de 25 Hz, con ventana Hann. |
| **Morada de trazo y punto · DWT Haar** | Energía de los detalles wavelet D4, D3 y D2 por muestra. Su banda es aproximada. |

El valor queda disponible al **cerrar** la ventana. Un disparo exige pasar
de debajo del umbral a alcanzarlo o superarlo. El código también espera
un segundo entre disparos aceptados. Cada método usa el umbral ya fijado
en el piloto.

In [ ]:
low, high = PROTOCOL["band_hz"]
print(f"Ventana: {WIDTH / FS:g} s · salto: {HOP / FS:g} s · banda STFT: [{low:g}, {high:g}) Hz")
print(f"Fondo: {PROTOCOL['background_frames']} ventanas previas")
print(f"Espera entre disparos aceptados: {PROTOCOL['holdoff_s']:g} s")
print("Umbrales:", " · ".join(f"{name} {value:.3f}" for name, value in thresholds.items()))
rows, curves = run_demo_detectors(samples, picks, thresholds)

## 4 · Cómo leer las gráficas del disparo

**Eje horizontal de ambos paneles:** segundos respecto a **P manual**.
El cero es cuando P llegó a la estación, no el inicio del sismo. A la
izquierda estamos antes de P; a la derecha, después. La línea vertical
de trazos marca P; la de puntos marca S.

- **A · Forma de onda original.** El eje vertical mide amplitud en cuentas
  digitales. Si dice **×10⁶**, una marca en 1 representa un millón de cuentas.
- **B · Respuesta de los detectores.** El eje vertical muestra el indicador
  de cada método dividido por su propio umbral. No tiene unidades.
  **1** es el nivel de disparo; **2** significa el doble del umbral.
  Los saltos grandes usan escala logarítmica: de 10 a 100 el valor
  se multiplica por diez, no aumenta en diez.

**Acercamiento.** El eje vertical es lineal y muestra solo el entorno del
umbral; los picos altos quedan fuera. Cada escalón empieza cuando se cierra
una ventana. Los puntos de color están sobre el nivel 1 para señalar
**cuándo** ocurrió el disparo asociado a P, no cuánto valía el indicador.

In [ ]:
display_figure(plot_demo(trace, picks, rows, curves))
display_figure(plot_crossings(rows, curves))

## 5 · Qué dice la tabla

- **Desfase respecto a P (s):** tiempo del disparo menos tiempo de P manual.
  Por ejemplo, **+0,070 s** son 70 milisegundos después de P.
- **Mediana por lote (ms):** tiempo típico de cálculo sobre los 89 s ya preparados.
- **P95 por lote (ms):** aproximadamente el 95 % de los tiempos medidos queda
  en ese valor o por debajo.

Para medir el desfase se toma el primer disparo dentro de la tolerancia del
piloto alrededor de P y antes de S. El cronómetro del lote excluye leer,
preparar y dibujar. Estos números describen **esta grabación**; no son el
tiempo de respuesta de un servicio de alerta.

La última celda abre el HTML con los resultados de esta misma ejecución.
Si no se abre, copia la ruta que imprime y pégala en el navegador.

In [ ]:
import pandas as pd

metrics = []
for row in rows:
    match = row["association"]
    metrics.append(
        {
            "Método": row["method"],
            "Desfase respecto a P (s)": f"{match['relative_to_p_s']:+.3f}"
            if match
            else "Sin asociación",
            "Mediana por lote (ms)": round(row["timing"]["median_ms"], 3),
            "P95 por lote (ms)": round(row["timing"]["p95_ms"], 3),
        }
    )
print(f"{TIMING['repetitions']} repeticiones tras {TIMING['warmups']} calentamientos.")
display(pd.DataFrame(metrics).set_index("Método"))

In [ ]:
report = export_demo_report(LIVE_DEST, trace, waveform_hash, picks, samples, rows, curves)
open_presentation(report)